# Phase 10 — Advanced Document Intelligence

## 1. Phase Overview

Phase 10 focused on improving the intelligence, safety, and decision quality of the VIGILOX document-processing pipeline.

By the end of Phase 9, VIGILOX already had:

```text
Document Upload
      ↓
Durable Job
      ↓
Background Worker
      ↓
PaddleOCR
      ↓
Groq Structured Extraction
      ↓
Validation
      ↓
Persistence
```

The next challenge was not simply extracting more data.

The important question became:

> Can the system distinguish between a document that was successfully processed and a document that is safe to trust?

Phase 10 therefore concentrated on advanced decision support around:

- document quality
- unsupported document handling
- duplicate detection
- extraction resilience
- confidence interpretation
- evidence quality
- finding normalization
- conservative automatic acceptance
- release safety

The goal was to make automatic decisions more reliable without inventing unsupported intelligence.

---

# 2. Objective

The main objective of Phase 10 was:

> Improve the reliability of machine decisions by combining extraction evidence, document quality, duplicate detection, domain classification, structured validation, and conservative routing rules.

The phase expanded the pipeline from:

```text
OCR
 ↓
LLM
 ↓
Schema
 ↓
Decision
```

into:

```text
Original Upload
      ↓
Fingerprint
      ↓
OCR
      ↓
Structured Extraction
      ↓
Evidence Validation
      ↓
Document-Type Support Check
      ↓
Date / Logical Validation
      ↓
Image Quality Assessment
      ↓
Finding Normalization
      ↓
Machine Decision
      ↓
AUTO_ACCEPT / REVIEW_REQUIRED / UNSUPPORTED
```

---

# 3. Problems Addressed

Several important limitations remained after the durable processing architecture was completed.

## 3.1 Successful Extraction Did Not Always Mean Correct Semantics

OCR can read text correctly while the structured extraction layer assigns the text to the wrong semantic field.

Example:

```text
Document contains:

Issue Date: 01/05/2025
Expiry Date: 01/05/2028

OCR:
Both dates correctly recognized

LLM:
01/05/2028 assigned as Issue Date
```

In this case:

```text
OCR Confidence → High
Semantic Correctness → Wrong
```

Therefore OCR confidence could not be treated as a probability that the extracted field was semantically correct.

---

## 3.2 Unsupported Documents Needed a Domain Outcome

A document that is not a:

```text
Security Guard Licence
ID Card
SIA Badge
```

should not be treated as:

```text
Provider Failure
OCR Failure
Infrastructure Failure
```

It needed its own explicit domain meaning.

---

## 3.3 Duplicate Uploads Could Waste Expensive Processing

Without duplicate protection:

```text
Same Document Uploaded Twice
        ↓
OCR Twice
        ↓
LLM Twice
        ↓
Duplicate Database Records
```

This wastes:

- CPU
- provider tokens
- processing time
- reviewer attention

---

## 3.4 Quality Signals Needed Calibration

A quality rule can become harmful if it produces false warnings on clean documents.

For example:

```text
Poorly calibrated threshold
       ↓
Clean documents flagged
       ↓
Review Queue inflation
       ↓
Reduced value of automation
```

Quality checks therefore needed measurement rather than arbitrary thresholds.

---

## 3.5 Findings Needed Consistent Semantics

The application accumulated findings from:

- OCR
- extraction
- evidence validation
- dates
- image quality
- document classification

These findings needed a normalized representation so the UI and decision engine could reason about them consistently.

---

# 4. Phase 10 Design Principles

Phase 10 followed several important principles.

## 4.1 Conservative Automatic Acceptance

Automatic acceptance is useful only when it remains safe.

The preferred trade-off was:

```text
Uncertain Correct Document
        ↓
Human Review
```

rather than:

```text
Incorrect Document
        ↓
AUTO_ACCEPT
```

The critical release invariant became:

```text
False AUTO_ACCEPT = 0
```

---

## 4.2 Do Not Invent Intelligence

VIGILOX does not generate unsupported values such as:

```text
Fraud Probability: 92%
Tamper Probability: 81%
AI Risk Score: 74%
Document Trust Score: 96%
```

unless such values are actually measured and validated.

The system reports concrete findings instead.

---

## 4.3 Quality and Classification Are Different

A poor-quality image does not automatically mean the document is unsupported.

Example:

```text
Blurry Guard Licence
```

is still:

```text
Supported Document
+
Image Quality Problem
```

not:

```text
Unsupported Document
```

---

## 4.4 Evidence Must Remain Inspectable

An extracted value should be tied to OCR evidence whenever possible.

The decision engine should not hide uncertainty behind a single numerical score.

---

# 5. Advanced Pipeline

The Phase 10 pipeline became:

```text
Upload
   ↓
Original-Byte SHA-256
   ↓
Duplicate Check
   ↓
OCR
   ↓
Structured Extraction
   ↓
Schema Validation
   ↓
Document Classification
   ↓
Evidence Validation
   ↓
Field Confidence
   ↓
Date / Logical Validation
   ↓
Image Quality Assessment
   ↓
Normalized Findings
   ↓
Machine Decision
   ↓
Persistence
```

This made document processing a multi-signal decision problem rather than a single LLM call.

---

# 6. Unsupported Document Handling

One of the most important Phase 10 changes was explicit unsupported-document handling.

An unsupported document is a valid processing outcome.

Typical semantics:

```text
Job Status:
COMPLETED

Document Type:
unknown

Supported:
false

Usable:
false

Retryable:
false

Effective Record:
none
```

The system completed the job because the pipeline successfully determined that the file was outside the supported document domain.

---

# 7. Why Unsupported Is Not FAILED

Consider:

```text
User uploads a restaurant receipt
```

If OCR works correctly and the classifier determines:

```text
Not a supported security credential
```

then the system has successfully answered the processing question.

Therefore:

```text
COMPLETED
```

is more accurate than:

```text
FAILED
```

The result is a domain outcome, not a technical failure.

---

# 8. Unsupported Document Rules

Unsupported documents:

- cannot become `AUTO_ACCEPTED`
- are not usable final records
- do not produce an effective credential record
- do not require provider retry
- remain auditable
- remain discoverable where appropriate
- do not automatically enter the standard Review Queue

This prevents unrelated uploads from polluting normal human-review work.

---

# 9. Neutral Unsupported Language

The system intentionally avoids describing unsupported documents using terms such as:

```text
Fraud
Suspicious
Tampered
Fake
Malicious
```

A document being outside the supported domain does not imply wrongdoing.

The appropriate description is simply:

```text
Unsupported
```

or:

```text
Unknown Document Type
```

---

# 10. Duplicate Detection

Phase 10 introduced source-level duplicate detection.

A SHA-256 fingerprint is computed from the exact original uploaded bytes.

```text
Original File Bytes
        ↓
SHA-256
        ↓
Source Fingerprint
```

The fingerprint is calculated before document preprocessing changes the image.

This is important because the fingerprint represents:

> the exact file submitted by the user

rather than an internally transformed image.

---

# 11. Duplicate Processing Flow

The duplicate flow became:

```text
Upload
   ↓
Compute SHA-256
   ↓
Search Existing Document
   ↓
Search Active Job
   ↓
Decision
```

Possible outcomes include:

```text
New Source
      ↓
Create Job

Existing Completed Document
      ↓
DUPLICATE_DOCUMENT

Existing Active Job
      ↓
DUPLICATE_IN_PROGRESS
```

---

# 12. Why Hash the Original Bytes

A source image may later be:

- resized
- normalized
- rotated
- converted
- contrast-adjusted

If hashing occurred after preprocessing, identical source documents could produce different fingerprints depending on pipeline configuration.

Using original bytes gives deterministic source identity.

---

# 13. Concurrent Duplicate Protection

Application-level duplicate checks alone are not enough.

This sequence is unsafe:

```text
Request A → check → no duplicate
Request B → check → no duplicate
Request A → insert
Request B → insert
```

Both requests could pass the check before either insert occurs.

This is a classic:

```text
check-then-insert race
```

---

# 14. PostgreSQL Duplicate Constraint

Phase 10 added database-level protection for active duplicate jobs.

Conceptually:

```text
Active Job Fingerprint
        ↓
Partial Unique Index
        ↓
Only One Active Processing Job
```

The database therefore remains the final concurrency authority.

This complements application-level detection.

---

# 15. Duplicate Reprocessing Policy

The default behavior avoids unnecessary duplicate processing.

A deliberate reprocess may still be permitted explicitly where required.

In that case:

```text
Original bytes remain identical
      ↓
SHA-256 remains identical
```

The fingerprint represents file identity, not processing attempt identity.

---

# 16. Fingerprint Privacy

The source fingerprint is useful internally for duplicate detection.

It is not normally exposed through public browser or API payloads.

This avoids turning internal integrity metadata into unnecessary public interface data.

---

# 17. Image Quality Intelligence

Phase 10 expanded deterministic image-quality assessment.

The goal was not to produce a vague quality percentage.

Instead, the system reports specific measurable conditions.

Implemented findings include:

```text
IMAGE_BLURRY
IMAGE_UNREADABLE
IMAGE_TOO_DARK
IMAGE_OVEREXPOSED
ROTATION_CONCERN
IMAGE_TOO_SMALL
```

Each finding describes a concrete condition.

---

# 18. Why Deterministic Quality Signals

Document quality can directly affect:

- OCR accuracy
- field extraction
- evidence matching
- reviewer confidence

Examples:

```text
Blur
→ characters merge

Low resolution
→ small text disappears

Dark exposure
→ text loses contrast

Rotation
→ OCR layout quality decreases
```

The quality subsystem therefore provides another signal for machine routing.

---

# 19. Quality Escalation

Quality findings may make the machine decision more conservative.

Example:

```text
Extraction otherwise acceptable
        +
IMAGE_BLURRY
        ↓
REVIEW_REQUIRED
```

However:

```text
Quality Finding
```

cannot transform an already unsafe result into:

```text
AUTO_ACCEPT
```

Quality signals can escalate risk but not erase another review requirement.

---

# 20. Quality Calibration

Thresholds were not accepted purely because they looked reasonable.

The checks were evaluated against the available document corpus and controlled degradations.

The purpose was to measure:

```text
Clean Documents
      ↓
How many false quality warnings?

Degraded Documents
      ↓
Does the intended signal trigger?
```

The shipped quality signals were calibrated so they did not introduce unnecessary warnings across the clean evaluation corpus used in the study.

---

# 21. False Positive Control

A quality detector that flags large numbers of clean documents would create:

```text
Review Queue Inflation
        ↓
Reviewer Fatigue
        ↓
Lower Automation Value
```

Therefore false-positive behavior was treated as a first-class evaluation criterion.

The implemented quality signals showed no clean-corpus false positives for the shipped checks in the calibration study.

---

# 22. Low Contrast Finding Decision

A possible signal:

```text
IMAGE_LOW_CONTRAST
```

was investigated.

However, the tested measurement did not prove sufficiently reliable across the available evaluation corpus.

Instead of shipping an unreliable finding, the feature was deliberately excluded.

This reflects an important research principle:

> A feature should not be shipped simply because it can be calculated.

---

# 23. Confidence Calibration

Phase 10 investigated whether field confidence could be interpreted as semantic correctness probability.

The original intuition might be:

```text
Confidence 98%
≈
98% probability field is correct
```

The evaluation did not support that interpretation.

---

# 24. OCR Confidence vs Semantic Correctness

Consider:

```text
OCR Line:
EXPIRY 11/08/2028

OCR Confidence:
0.99

Structured Extraction:
issue_date = 11/08/2028
```

The OCR engine may be almost perfectly certain about the characters.

But the semantic assignment is wrong.

Therefore:

```text
OCR Confidence
≠
Semantic Field Correctness Probability
```

---

# 25. Final Confidence Interpretation

VIGILOX therefore treats field confidence as:

> OCR and evidence support strength

rather than:

> probability that the extracted semantic field is correct

This distinction is reflected in both the UI and documentation.

---

# 26. No Document-Level Confidence Percentage

Because field confidence was not calibrated as semantic correctness probability, the system avoided deriving a generic:

```text
Document Confidence: 94%
```

Such a number would look precise without having a valid statistical interpretation.

VIGILOX instead exposes:

- field confidence
- evidence
- quality findings
- validation findings
- machine decision

These signals remain individually interpretable.

---

# 27. Evidence Validation

Evidence validation remained a central component of Phase 10.

For a structured field:

```text
Field Value
     ↓
Expected OCR Evidence
     ↓
Evidence Match
     ↓
Confidence / Findings
```

The system attempts to verify that extracted structured values can be supported by OCR observations.

---

# 28. Evidence Does Not Prove Semantics

Evidence validation provides an important guarantee:

```text
The extracted value exists in or is supported by OCR
```

It does not automatically prove:

```text
The value was assigned to the correct semantic field
```

This distinction is why evidence, validation, and semantic extraction remain separate stages.

---

# 29. Date Validation

Security credentials often contain date relationships such as:

```text
Issue Date
Expiry Date
Date of Birth
```

Phase 10 continued strengthening deterministic date validation.

Example checks include:

```text
Expiry date should not logically precede issue date

Expired credentials should be identifiable

Date formats should normalize where possible

Missing critical dates should influence machine decision
```

These rules complement the LLM rather than asking the LLM to decide every logical relationship.

---

# 30. Logical Validation

Structured extraction can produce syntactically valid JSON while still being logically inconsistent.

Example:

```json
{
  "issue_date": "2028-04-10",
  "expiry_date": "2025-04-10"
}
```

Pydantic may accept both values as valid date strings.

Domain validation must still detect:

```text
expiry_date < issue_date
```

Therefore Phase 10 maintained a distinction between:

```text
Schema Validity
```

and:

```text
Domain Validity
```

---

# 31. Structured Extraction Resilience

LLM structured extraction can fail in several ways:

```text
Malformed output
Schema mismatch
Provider error
Model unavailable
Timeout
Rate limit
```

Phase 10 tightened how these failures were classified and recovered.

---

# 32. Bounded Structured Output Recovery

Structured output recovery uses a bounded attempt count.

Representative configuration:

```text
VIGILOX_EXTRACTION_ATTEMPTS=3
```

The system does not retry indefinitely when a model repeatedly fails to produce schema-compatible output.

---

# 33. Provider Retry Boundaries

The system distinguishes:

```text
Structured Output Failure
```

from:

```text
Provider Infrastructure Failure
```

Example:

```text
Malformed structured response
→ extraction-level recovery

429 rate limit
→ job-level retry

5xx provider error
→ job-level retry

connection failure
→ job-level retry
```

This separation avoids wasting OCR work unnecessarily while still keeping the total execution budget bounded.

---

# 34. Model Configuration

The primary extraction model is configurable:

```env
VIGILOX_GROQ_MODEL=openai/gpt-oss-20b
```

This avoids hard-coding provider lifecycle decisions into application code.

If a model is retired, configuration can be updated without redesigning the extraction service.

---

# 35. Model Fallback

An optional fallback-model mechanism was introduced with deliberately narrow semantics.

A fallback may be used for a specific model-availability condition such as:

```text
Primary model does not exist / is unavailable as a configured model
```

It is not intended to activate automatically for:

```text
Rate limit
Temporary provider outage
Formatting error
Connection failure
```

Switching models in those situations could change extraction behavior for reasons unrelated to model correctness.

---

# 36. Same Schema for Fallback

If a fallback model is used, it must still obey:

```text
Same Structured Schema
Same Validation Rules
Same Evidence Requirements
```

The fallback changes the provider model, not the document contract.

---

# 37. Finding Normalization

Phase 10 introduced more disciplined finding normalization.

Multiple subsystems can generate issues:

```text
OCR
Extraction
Evidence
Dates
Quality
Classification
```

Without normalization, the frontend could receive inconsistent shapes and naming.

The desired model became:

```text
Finding
├── code
├── category
├── severity / routing meaning
├── field reference where applicable
└── safe human-readable message
```

This provides a stable interface between intelligence services and UI rendering.

---

# 38. Finding Codes

Examples include:

```text
IMAGE_BLURRY
IMAGE_UNREADABLE
IMAGE_TOO_DARK
IMAGE_OVEREXPOSED
ROTATION_CONCERN
IMAGE_TOO_SMALL
```

Other validation and extraction findings follow the same normalized pattern.

The important principle is that machine-readable codes remain stable even if display wording changes.

---

# 39. Decision Engine

The Phase 10 machine decision combines multiple classes of signal.

Conceptually:

```text
Supported Document?
       ↓
Critical Fields Present?
       ↓
Evidence Sufficient?
       ↓
Validation Clean?
       ↓
Image Quality Acceptable?
       ↓
Decision
```

Possible decision outcomes include:

```text
AUTO_ACCEPT
REVIEW_REQUIRED
```

while unsupported documents follow their separate domain outcome.

---

# 40. Automatic Acceptance Safety

The most important rule is:

> Automatic acceptance must be conservative.

A document should not be automatically accepted simply because:

```text
LLM returned JSON
```

or because:

```text
OCR confidence is high
```

Automatic acceptance requires the complete decision criteria to be satisfied.

---

# 41. False AUTO_ACCEPT

The highest-risk evaluation error is:

```text
Incorrect Document
        ↓
AUTO_ACCEPT
```

because this bypasses human review.

For that reason the release-critical metric remained:

```text
False AUTO_ACCEPT = 0
```

This metric is more operationally meaningful than optimizing only for the percentage of documents automatically accepted.

---

# 42. Human Review as a Safety Mechanism

Routing more uncertain documents to human review is not considered system failure.

It is an intentional safety design.

The preferred hierarchy is:

```text
High Confidence + Strong Evidence + Valid Logic
        ↓
AUTO_ACCEPT

Uncertainty
        ↓
REVIEW_REQUIRED

Unsupported Domain
        ↓
UNSUPPORTED
```

---

# 43. Evaluation Corpus

The advanced intelligence work was evaluated against the VIGILOX labelled corpus.

The historical benchmark contained:

```text
63 documents
```

which exceeded the original project requirement of at least 50 labelled documents.

The corpus included the supported credential categories and controlled test conditions used throughout the project.

---

# 44. Historical Evaluation Results

Historical benchmark results included:

```text
Document Type Accuracy           100%

Exact Field Accuracy             95.92%

Normalized Field Accuracy        98.64%

Known-Field Normalized Accuracy  98.49%

Fully Correct Documents          93.65%

False AUTO_ACCEPT                0
```

These values represent the historical verified evaluation baseline.

---

# 45. Critical Field Evaluation

Critical-field accuracy required special attention.

An earlier evaluation definition omitted the production-critical:

```text
issuer
```

field for relevant document types.

The earlier reported result was:

```text
99.40%
167 / 168
```

After aligning evaluation with the authoritative production critical-field definition, the corrected historical result became:

```text
99.05%
208 / 210
```

---

# 46. Why the Metric Changed

The change:

```text
99.40%
     ↓
99.05%
```

was not a model regression.

The extraction results did not suddenly become worse.

Instead:

```text
Old Metric Definition
        ↓
Missing Critical Fields
        ↓
Metric Corrected
        ↓
Production-Aligned Definition
```

This is a measurement-definition correction.

---

# 47. Single Source of Truth for Critical Fields

To prevent future metric drift, evaluation was aligned with the authoritative production critical-field definition.

This avoids maintaining:

```text
Production Critical Fields
```

and:

```text
Evaluation Critical Fields
```

as separate lists.

Duplicated definitions eventually diverge.

---

# 48. Evaluation Integrity

The evaluation framework was treated as part of the system, not just an external report.

It needed to answer:

```text
What was measured?
Which fields count?
Which documents were evaluated?
How were values normalized?
Which decision was considered safe?
```

This increased confidence that improvements were real rather than artifacts of inconsistent metrics.

---

# 49. Quality Calibration Results

The image-quality work evaluated clean and intentionally degraded documents.

The shipped quality signals were selected because their tested behavior was useful without introducing clean-corpus false positives in the available calibration set.

This meant quality findings could safely contribute to conservative routing.

---

# 50. Clean Corpus False Positives

For the shipped quality checks, the calibration work showed:

```text
Clean corpus false positives: 0
```

for the signals that were retained.

This was important because a quality system that flags clean credentials would reduce reviewer trust.

---

# 51. Unsupported Document Testing

Unsupported documents were tested to ensure they did not accidentally enter normal success logic.

Important assertions included:

```text
Supported = false

Usable = false

No effective record

Cannot auto-accept

No unnecessary provider retry

No normal review-queue pollution
```

---

# 52. Duplicate Testing

Duplicate tests needed to cover both sequential and concurrent behavior.

Examples:

```text
Upload completed document again
→ duplicate document

Upload while same source already processing
→ duplicate in progress

Concurrent same-file uploads
→ database prevents two active processing jobs
```

This ensured duplicate handling was not only a UI convenience.

---

# 53. Extraction Resilience Testing

The extraction-resilience test suite covered cases such as:

```text
Structured response recovery
Provider timeout
Rate limit
Model error
Retry bounds
Fallback behavior
Lease execution budget
```

The goal was to ensure every retry layer remained bounded.

---

# 54. Lease Budget Relationship

Extraction retries and provider timeouts influence the maximum possible worker execution time.

The worker lease therefore cannot be configured independently of extraction behavior.

Conceptually:

```text
OCR Maximum
      +
Provider Request Budget
      +
Structured Retry Budget
      +
Safety Headroom
      <
Worker Lease
```

This relationship was validated in tests.

---

# 55. Advanced Intelligence Is Not One Model

A major Phase 10 conclusion was that document intelligence should not be represented as:

```text
LLM
```

Instead, the intelligence system is:

```text
OCR
+
Structured Extraction
+
Evidence Validation
+
Quality Measurement
+
Logical Validation
+
Duplicate Detection
+
Domain Classification
+
Decision Rules
```

The model is only one component.

---

# 56. Machine Learning vs Deterministic Logic

VIGILOX deliberately uses different techniques for different problems.

For example:

```text
OCR
→ PaddleOCR

Semantic Structured Extraction
→ LLM

Schema Validation
→ Pydantic

Duplicate Identity
→ SHA-256

Date Relationships
→ deterministic rules

Image Quality
→ deterministic measurements

Concurrency Safety
→ PostgreSQL constraints
```

This avoids using AI where deterministic logic is more reliable.

---

# 57. No Fraud Detection Claim

VIGILOX does not claim to perform forensic fraud detection.

It may identify:

```text
Extraction issues
Evidence inconsistencies
Expiry
Image quality problems
Logical inconsistencies
Unsupported document type
```

These are not equivalent to proving fraud or tampering.

The terminology remains intentionally conservative.

---

# 58. No Tamper Detection Claim

Likewise, image quality problems such as:

```text
blur
dark exposure
small dimensions
rotation
```

do not prove tampering.

The application reports what it measures rather than converting image properties into unsupported forensic conclusions.

---

# 59. Review Priority

Normalized findings and machine decision rules can contribute to reviewer prioritization.

However, review priority is an operational routing tool.

It is not presented as a probabilistic risk score.

The distinction remains:

```text
Priority
→ workflow ordering

Risk probability
→ statistical claim
```

VIGILOX uses the first without pretending to have the second.

---

# 60. Final Record Safety

Phase 10 reinforced the separation between:

```text
Processing Completed
```

and:

```text
Record Usable
```

A document can complete processing while still being:

```text
PENDING_REVIEW
REJECTED
UNSUPPORTED
```

Only explicitly usable final states can provide an effective record.

---

# 61. Relationship with Human Review

Advanced intelligence does not replace the human review layer.

Instead:

```text
Machine Intelligence
        ↓
Confident Result
        ↓
Automatic Acceptance

or

Machine Intelligence
        ↓
Uncertainty
        ↓
Human Review
```

Phase 10 therefore improves review routing as much as it improves automatic processing.

---

# 62. Key Implementation Areas

Phase 10 work was concentrated around areas such as:

```text
backend/app/services/
database/repositories
tests/intelligence/
evaluation/
scripts/evaluation/
scripts/development/
```

Representative responsibilities included:

- quality analysis
- extraction resilience
- duplicate detection
- domain classification
- confidence calibration
- decision rules
- finding normalization
- evaluation metrics

---

# 63. Research and Calibration Scripts

Phase 10 relied on scripts and evaluation tools for:

```text
Image degradation experiments
Quality threshold analysis
Extraction latency study
Confidence calibration
Evaluation metric generation
Synthetic fixture generation
Regression verification
```

This is important for the research repository because the phase was driven by measured behavior rather than only feature implementation.

---

# 64. What Phase 10 Did Not Do

Phase 10 deliberately did not:

- generate a fake overall document confidence score
- claim fraud detection
- claim tamper detection
- use OCR confidence as semantic correctness probability
- auto-accept unsupported documents
- retry unsupported documents as provider failures
- allow duplicate active processing without database protection
- ship every experimental quality metric
- maximize auto-acceptance at the expense of safety

---

# 65. Lessons Learned

## 65.1 High OCR Confidence Does Not Mean Correct Semantics

Character recognition and field interpretation are different problems.

A system must evaluate them separately.

---

## 65.2 Conservative Automation Is Better Than Unsafe Automation

A lower automatic acceptance rate can be acceptable if it keeps:

```text
False AUTO_ACCEPT = 0
```

The purpose is reducing human workload safely, not maximizing the number of documents processed without review.

---

## 65.3 Unsupported Is a Valid Answer

A system should be able to say:

```text
I processed this successfully,
but it is outside my supported domain.
```

without treating that as infrastructure failure.

---

## 65.4 Quality Metrics Must Be Calibrated

A numerical image measurement is not automatically useful.

It should only become a product finding after its behavior is tested.

---

## 65.5 Database Constraints Matter for Intelligence Cost

Duplicate protection is not only a data-quality issue.

It protects:

- OCR CPU
- LLM quota
- reviewer workload
- database cleanliness

The database is therefore part of the intelligence system's efficiency.

---

## 65.6 Metric Definitions Are Part of the Product

A benchmark can look excellent while measuring the wrong definition.

Evaluation logic must share authoritative production definitions wherever possible.

---

## 65.7 Findings Are Better Than Unsupported Scores

Concrete output such as:

```text
IMAGE_BLURRY
EXPIRY_DATE_INVALID
MISSING_CRITICAL_FIELD
```

is more defensible than:

```text
Risk = 73%
```

when no calibrated statistical model exists.

---

# 66. Phase 10 Deliverables

Phase 10 produced:

- advanced image-quality assessment
- calibrated quality thresholds
- unsupported-document domain handling
- original-byte SHA-256 fingerprinting
- duplicate document detection
- active duplicate-job protection
- PostgreSQL concurrency constraint
- extraction resilience improvements
- bounded structured-output retries
- provider timeout controls
- configurable Groq model
- controlled model fallback
- confidence calibration
- clearer confidence semantics
- finding normalization
- production-aligned critical-field evaluation
- conservative decision routing
- explicit false-auto-accept safety invariant

---

# 67. Architecture After Phase 10

```text
                    ┌──────────────────────┐
                    │    Original Upload   │
                    └──────────┬───────────┘
                               │
                               ▼
                    ┌──────────────────────┐
                    │ SHA-256 Fingerprint  │
                    └──────────┬───────────┘
                               │
                               ▼
                    ┌──────────────────────┐
                    │ Duplicate Detection  │
                    └──────────┬───────────┘
                               │
                               ▼
                    ┌──────────────────────┐
                    │      PaddleOCR       │
                    └──────────┬───────────┘
                               │
                               ▼
                    ┌──────────────────────┐
                    │  Groq Extraction     │
                    └──────────┬───────────┘
                               │
                ┌──────────────┼──────────────┐
                │              │              │
                ▼              ▼              ▼
        ┌──────────────┐ ┌──────────────┐ ┌──────────────┐
        │   Evidence   │ │    Quality   │ │ Date / Logic │
        │  Validation  │ │  Assessment  │ │  Validation  │
        └──────┬───────┘ └──────┬───────┘ └──────┬───────┘
               │                │                │
               └────────────────┼────────────────┘
                                │
                                ▼
                     ┌────────────────────┐
                     │ Normalized Findings│
                     └─────────┬──────────┘
                               │
                               ▼
                     ┌────────────────────┐
                     │ Machine Decision   │
                     └─────────┬──────────┘
                               │
             ┌─────────────────┼─────────────────┐
             │                 │                 │
             ▼                 ▼                 ▼
       AUTO_ACCEPT      REVIEW_REQUIRED     UNSUPPORTED
```

---

# 68. Final Outcome

Phase 10 moved VIGILOX from:

```text
A document extraction system
```

toward:

```text
A conservative document intelligence system
```

The pipeline no longer relied only on whether OCR and the LLM returned values.

It considered:

```text
Source Identity
Document Support
Evidence
Image Quality
Logical Consistency
Critical Fields
Extraction Resilience
Duplicate State
```

before making a machine decision.

The result was a safer and more explainable document-processing workflow.

The guiding principle became:

> Extract what can be extracted, verify what can be verified, and route uncertainty instead of hiding it.

---

# 69. Phase 10 Summary

| Area | Result |
|---|---|
| Unsupported Document Handling | Implemented |
| Original-Byte SHA-256 Fingerprinting | Implemented |
| Duplicate Document Detection | Implemented |
| Concurrent Duplicate Protection | Implemented |
| Image Quality Assessment | Implemented |
| Quality Calibration | Completed |
| Clean-Corpus Quality False Positives | 0 for shipped signals |
| Confidence Calibration | Completed |
| OCR vs Semantic Confidence Separation | Established |
| Finding Normalization | Implemented |
| Extraction Resilience | Implemented |
| Bounded Structured Output Recovery | Implemented |
| Configurable Groq Model | Implemented |
| Controlled Model Fallback | Implemented |
| Production-Aligned Critical Fields | Implemented |
| Corrected Historical Critical Accuracy | 99.05% (208 / 210) |
| Historical Document-Type Accuracy | 100% |
| Historical Normalized Field Accuracy | 98.64% |
| Historical Fully Correct Documents | 93.65% |
| Historical False AUTO_ACCEPT | 0 |
| Unsupported Fraud/Tamper Claims | Avoided |

---

**Next:** `Phase 11 — Production Hardening, Security, Deployment Architecture and Observability`